# Module 05 — A2A (Agent-to-Agent) Protocol

**Released:** v1.8.0 → v1.13.0 (Jan–Apr 2026)

A2A makes agent-to-agent delegation a first-class primitive: agents can expose themselves as standards-compliant HTTP servers (other CrewAI agents or anything speaking A2A can call them), and/or consume remote A2A agents as delegable tools.

Two sides of the protocol:

- **Server** — `A2AServerConfig` declares an agent as an A2A endpoint. `agent.to_agent_card(url)` emits the agent card that clients discover at `.well-known/agent-card.json`.
- **Client** — `A2AClientConfig` points at a remote endpoint. The agent's LLM decides per-task whether to handle locally or delegate.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## 1. Server side — declare and inspect

Build a worker agent with `A2AServerConfig`, then render its agent card.

In [ ]:
from showcase.flows.a2a_flow import build_server_agent

worker = build_server_agent()
card = worker.to_agent_card("http://localhost:8765")
print(card.name, "—", card.description)
print("streaming:", card.capabilities.streaming)
print("skills:", [s.name for s in card.skills])

## 2. Serve the worker (outside the notebook)

The A2A server runs as an HTTP process. In practice you'd host this behind uvicorn or deploy via AMP. A minimal local runner pattern:

```python
# script: serve_worker.py
from crewai.a2a.server import build_asgi_app  # enterprise convenience; see docs
from showcase.flows.a2a_flow import build_server_agent

app = build_asgi_app(build_server_agent())  # FastAPI/Starlette app
# Run: uvicorn serve_worker:app --port 8765
```

For production: see the [A2A on AMP docs](https://docs.crewai.com/en/enterprise/features/a2a) for auth, gRPC, and scaling.

> The client demo below uses `fail_fast=False`, so this notebook runs with or without a live server.

## 3. Client side — delegate

The coordinator points `A2AClientConfig.endpoint` at the worker's agent card URL. The LLM decides whether to delegate or handle locally. With `fail_fast=False`, unreachable endpoints are skipped and the agent degrades to local execution — perfect for dev loops.

In [ ]:
from showcase.flows.a2a_flow import A2AClientFlow

flow = A2AClientFlow()
flow.kickoff(inputs={"question": "What changed in the A2A client API between v1.8 and v1.13?"})
print(flow.state.answer)

## 4. Multiple remote workers

Pass a list of `A2AClientConfig` and the LLM picks the right specialist per task:

```python
Agent(
    ...,
    a2a=[
        A2AClientConfig(endpoint="https://research.example.com/.well-known/agent-card.json"),
        A2AClientConfig(endpoint="https://data.example.com/.well-known/agent-card.json"),
    ],
)
```

## Try this

- Boot the worker server (step 2) in a separate terminal, then re-run the client flow — watch the trace show remote delegation instead of local execution.
- Add `auth=BearerTokenAuth(token=...)` to the client config and secure the server with `SimpleTokenAuth`.
- Switch updates to `PollingConfig(interval=2.0)` for flaky networks.